# Logistic Regression — Training and Analysis

Trains a logistic-regression baseline on the **leakage-corrected** CICEVSE2024
network-traffic dataset, out-of-core via `SGDClassifier(loss="log_loss")`.

| | |
|---|---|
| **Dataset** | CICEVSE2024 network traffic — 15 classes (14 attacks + benign) |
| **Features** | 60, after dropping six capture timestamps and `src_port` |
| **Task** | Multiclass only — only 82 benign flows exist, so binary is meaningless |
| **Headline metric** | Macro-F1 |

> Alongside the SVM, this is the linear control in the leakage finding. Both linear
> models sit around 0.41–0.44 macro-F1 with the leak removed, and both barely moved
> when it was removed — unlike the tree models, which fell sharply.

In [ ]:
import os

import joblib

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.linear_model import SGDClassifier

from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from tqdm.notebook import tqdm



# Plotting config

%matplotlib inline

plt.rcParams['figure.dpi'] = 100

sns.set_theme(style='whitegrid', palette='deep')

## 1. Load Data

Load the preprocessed train, validation, and test datasets.

In [ ]:
# Adjust path assuming the notebook runs from the project root or src/models/log_reg

if os.path.exists('../../../data/processed'):

    DATA_DIR = '../../../data/processed'

elif os.path.exists('data/processed'):

    DATA_DIR = 'data/processed'

else:

    DATA_DIR = '../data/processed'



print(f"Using data directory: {DATA_DIR}")



# Load targets fully (small enough to fit in RAM)

y_train = pd.read_csv(os.path.join(DATA_DIR, "y_train.csv"))

y_val = pd.read_csv(os.path.join(DATA_DIR, "y_val.csv"))

y_test = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv"))



# Load a small sample of X_train for visualisation (full dataset is 2.4 GB)

X_train_sample = pd.read_csv(os.path.join(DATA_DIR, "X_train.csv"), nrows=50000)

X_val = pd.read_csv(os.path.join(DATA_DIR, "X_val.csv"))

X_test = pd.read_csv(os.path.join(DATA_DIR, "X_test.csv"))



y_train_multi = y_train["Label_Multiclass"].values.ravel()

y_val_multi = y_val["Label_Multiclass"].values.ravel()

y_test_multi = y_test["Label_Multiclass"].values.ravel()



print("Data loaded successfully!")

print(f"y_train shape: {y_train.shape}")

print(f"X_train_sample shape (for visualisation): {X_train_sample.shape}")

print(f"X_val shape: {X_val.shape}")

print(f"X_test shape: {X_test.shape}")

In [ ]:
# ── Leakage guard (issues #46, #58) ──────────────────────────────────────────
# The published CICEVSE2024 pipeline keeps six absolute capture timestamps that
# identify the *recording* rather than the traffic: a decision tree on
# `bidirectional_first_seen_ms` alone scores 1.0000 on the 15-class target.
# `src_port` is a weaker version of the same thing — ephemeral ports are
# allocated near-sequentially, so they band per capture (0.4028 alone).
#
# Both are dropped by `preprocess.py`. This cell fails loudly if you are sitting
# on data built by an older version of the pipeline, because every number below
# would be meaningless.
LEAKING = [
    "bidirectional_first_seen_ms", "bidirectional_last_seen_ms",
    "src2dst_first_seen_ms", "src2dst_last_seen_ms",
    "dst2src_first_seen_ms", "dst2src_last_seen_ms",
    "src_port",
]

# Notebooks load either the full matrix or a sample frame for visualisation.
_frame = next((globals()[n] for n in ("X_train", "X_train_sample") if n in globals()), None)
assert _frame is not None, "Run the data-loading cell above first."

present = [c for c in LEAKING if c in _frame.columns]
assert not present, (
    f"Leaking columns found: {present}. Regenerate with: make data-process"
)
print(f"Leakage guard passed — {_frame.shape[1]} features, none of them capture fingerprints.")

# Relative timing is behavioural and is deliberately retained.
kept = [c for c in _frame.columns if c.endswith("_duration_ms") or c.endswith("_piat_ms")]
print(f"Relative timing features retained: {len(kept)}")


## 1.1 Dataset Overview

Quick inspection of the training data: shape, data types, summary statistics, and missing value check.

In [ ]:
total_train = len(y_train)

print(f"Full X_train rows : {total_train:,}")

print(f"X_val rows        : {len(X_val):,}")

print(f"X_test rows       : {len(X_test):,}")

print(f"\nNumber of features: {X_train_sample.shape[1]}")

print(f"\nData types:\n{X_train_sample.dtypes.value_counts()}")

print(f"\nMissing values per column (if any):")

missing = X_train_sample.isnull().sum()

missing_cols = missing[missing > 0]

if len(missing_cols) == 0:

    print("  None — all features are clean.")

else:

    print(missing_cols)



print("\nSummary Statistics (first 10 features):")

X_train_sample.iloc[:, :10].describe().round(3)

## 1.2 Train / Validation / Test Split Sizes

In [ ]:
split_sizes = pd.DataFrame({

    'Split': ['Train', 'Validation', 'Test'],

    'Samples': [total_train, len(X_val), len(X_test)]

})

split_sizes['Percentage'] = (split_sizes['Samples'] / split_sizes['Samples'].sum() * 100).round(1)



fig, ax = plt.subplots(figsize=(8, 3))

bars = ax.barh(split_sizes['Split'], split_sizes['Samples'], color=['#2196F3', '#FF9800', '#4CAF50'])

for bar, pct in zip(bars, split_sizes['Percentage']):

    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,

            f'{bar.get_width():,.0f} ({pct}%)', va='center', fontsize=11)

ax.set_xlabel('Number of Samples')

ax.set_title('Train / Validation / Test Split Sizes')

plt.tight_layout()

plt.show()

## 1.3 Visualize Class Distributions

In [ ]:
multi_counts = y_train['Label_Multiclass'].value_counts()

colors = sns.color_palette('viridis', len(multi_counts))



fig, ax = plt.subplots(figsize=(12, 6))

ax.barh(multi_counts.index, multi_counts.values, color=colors)

ax.set_title('Multiclass Distribution (Train)', fontsize=14)

ax.set_xlabel('Count')

ax.set_ylabel('Attack Type')

for i, val in enumerate(multi_counts.values):

    ax.text(val + 1000, i, f'{val:,}', va='center', fontsize=9)



plt.tight_layout()

plt.show()



print("\nMulticlass label counts:")

print(multi_counts.to_string())

## 1.4 Feature Correlation Heatmap

In [ ]:
top_features = X_train_sample.var().nlargest(30).index.tolist()

corr_matrix = X_train_sample[top_features].corr()



plt.figure(figsize=(14, 12))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,

            square=True, linewidths=0.5, fmt='.1f',

            cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'})

plt.title('Feature Correlation Heatmap (Top 30 by Variance)', fontsize=14)

plt.xticks(rotation=45, ha='right', fontsize=8)

plt.yticks(fontsize=8)

plt.tight_layout()

plt.show()

## 1.5 Feature Distribution Box Plots

In [ ]:
top10 = X_train_sample.var().nlargest(10).index.tolist()



fig, axes = plt.subplots(2, 5, figsize=(20, 8))

axes = axes.flatten()



for i, col in enumerate(top10):

    sample = X_train_sample[col].sample(n=min(10000, len(X_train_sample)), random_state=42)

    axes[i].boxplot(sample.values, vert=True, patch_artist=True,

                    boxprops=dict(facecolor='#AB47BC', alpha=0.7))

    axes[i].set_title(col, fontsize=9, fontweight='bold')

    axes[i].tick_params(axis='x', labelbottom=False)



plt.suptitle('Top 10 Features by Variance — Box Plots (Scaled Data)', fontsize=14, y=1.02)

plt.tight_layout()

plt.show()

## 2. Train Logistic Regression (Out-of-Core)



We use `SGDClassifier` with `loss='log_loss'` which mathematically replicates Logistic Regression but supports `partial_fit()` for Out-of-Core learning on the 2.4 GB dataset.



We pre-compute class weights for balanced training to handle the extreme class imbalance.

In [ ]:
# Pre-compute class weights

classes_multi = np.array(sorted(y_train['Label_Multiclass'].unique()))

weights_m = compute_class_weight('balanced', classes=classes_multi, y=y_train_multi)

class_weight_multi = dict(zip(classes_multi, weights_m))



print(f"Classes ({len(classes_multi)}): {classes_multi}")

print(f"\nClass weights:")

for cls, w in sorted(class_weight_multi.items(), key=lambda x: x[1], reverse=True):

    print(f"  {cls:>25s}: {w:.4f}")

In [ ]:
# Initialize model

model_multi = SGDClassifier(loss='log_loss', random_state=42, class_weight=class_weight_multi)



# Training parameters

chunk_size = 100000

total_samples = len(y_train)



print(f"Training Logistic Regression (Out-of-Core) on {total_samples:,} samples...")

print(f"Chunk size: {chunk_size:,}")



X_train_path = os.path.join(DATA_DIR, "X_train.csv")

y_train_path = os.path.join(DATA_DIR, "y_train.csv")



X_chunker = pd.read_csv(X_train_path, chunksize=chunk_size)

y_chunker = pd.read_csv(y_train_path, chunksize=chunk_size)



with tqdm(total=total_samples, desc="Training Logistic Regression") as pbar:

    for X_chunk, y_chunk in zip(X_chunker, y_chunker):

        model_multi.partial_fit(X_chunk, y_chunk["Label_Multiclass"], classes=classes_multi)

        pbar.update(len(X_chunk))



print("\nTraining complete!")

## 3. Evaluation

In [ ]:
def evaluate_model(model, X, y, title_prefix=""):

    preds = model.predict(X)

    

    acc = accuracy_score(y, preds)

    f1_macro = f1_score(y, preds, average='macro', zero_division=0)

    f1_weighted = f1_score(y, preds, average='weighted', zero_division=0)

    

    print(f"--- {title_prefix} ---")

    print(f"Accuracy       : {acc:.4f}")

    print(f"Macro F1-Score : {f1_macro:.4f}")

    print(f"Weighted F1    : {f1_weighted:.4f}")

    print(f"\nClassification Report:")

    print(classification_report(y, preds, zero_division=0))

    

    cm = confusion_matrix(y, preds)

    fig_size = max(8, len(np.unique(y)) * 0.8)

    plt.figure(figsize=(fig_size + 2, fig_size))

    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples")

    plt.title(f"{title_prefix} Confusion Matrix")

    plt.ylabel('Actual')

    plt.xlabel('Predicted')

    plt.tight_layout()

    plt.show()

    return preds

In [ ]:
_ = evaluate_model(model_multi, X_val, y_val_multi, title_prefix="Logistic Regression — Validation Set")

In [ ]:
y_pred_test = evaluate_model(model_multi, X_test, y_test_multi, title_prefix="Logistic Regression — Test Set")

## 4. Save Model and Predictions

In [ ]:
# Resolve save directories

if os.path.exists('../../../saved_models'):

    SAVE_DIR = '../../../saved_models'

    PREDS_DIR = '../../../predictions'

elif os.path.exists('saved_models'):

    SAVE_DIR = 'saved_models'

    PREDS_DIR = 'predictions'

else:

    SAVE_DIR = '../saved_models'

    PREDS_DIR = '../predictions'



os.makedirs(SAVE_DIR, exist_ok=True)

os.makedirs(PREDS_DIR, exist_ok=True)



# Save model

model_path = os.path.join(SAVE_DIR, "logreg_model_multiclass.pkl")

joblib.dump(model_multi, model_path)

print(f"Model saved to {model_path}")



# Save predictions

preds_df = pd.DataFrame({

    'Prediction_Multiclass': y_pred_test,

    'y_true': y_test_multi,

    'y_pred': y_pred_test

})

preds_path = os.path.join(PREDS_DIR, "logreg_preds_multiclass.csv")

preds_df.to_csv(preds_path, index=False)

print(f"Predictions saved to {preds_path}")



print("\nLogistic Regression pipeline complete!")

---







## Where the canonical numbers live







This notebook is for exploration. Every figure quoted in the write-up comes from



`src/evaluation/run_experiments.py`, which re-runs the split *and* model



initialisation across seeds 42 / 1337 / 2024 and reports mean ± standard



deviation. A single-seed result from this notebook will differ, and should not



be quoted on its own.







```bash



python3 src/evaluation/run_experiments.py --raw-dir data/raw --feature-set extended



```







| Reference | Location |



|---|---|



| Multi-seed results | `evaluation_results/multiseed/MULTISEED_RESULTS.md` |



| Full leakage audit | `docs/leakage_audit_results.md` |



| Feature-set diff vs. the reference implementation | `docs/dataset_feature_engineering.md` §3.4 |







**Headline for this dataset is macro-F1, not accuracy.** With `ICMP_Fragmentation`



at 28 flows and `Benign` at 82, accuracy tracks the volumetric floods and hides



the reconnaissance classes almost entirely — the corrected Random Forest scores



0.8679 accuracy against 0.5582 macro-F1 on the same predictions.